In [ ]:
# ============================================================
# Core imports + config
# ============================================================
import os, sys, gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from scipy.ndimage import gaussian_filter
from scipy.integrate import trapezoid
import seaborn as sns
import pickle
from tqdm.auto import tqdm

dt = 0.005
t_pre, t_post = 0.3, 0.3

SESSION_TYPE = "playback"  # "playback" or "eTheremin" session type for load_pickled_ss

USE_MACRO_EPOCH = True   # True = contiguous Condition blocks, False = peri-event windows
CEBRA_DISTANCE = "euclidean"  # "cosine" or "euclidean"
CEBRA_ARCH = "offset10-model" if CEBRA_DISTANCE == "cosine" else "offset10-model-mse"
TAU_SHIFT = 6       # label lag in bins (6 = 30ms at dt=0.005, 0 = disabled)
LIE_METHOD = "lstsq"  # "lstsq" = OLS + skew-symmetrize (recommended); "pytorch" = constrained optimization
MIN_EPOCH_DUR = 2.0      # minimum macro-epoch duration (seconds)

NAS = r"\\129.199.81.18\data5\eTheremin"

mpl.rcdefaults()
plt.rcParams.update({
    'font.size': 7, 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
print("Imports ready.")

In [ ]:
# ============================================================
# Load spike-sorted Skieur data
# ============================================================

def load_pickled_ss(file_prefix, session_type, dt):
    data_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_data_ss")
    feat_path = os.path.join(NAS, f"{file_prefix}_{session_type}_{dt}_feature_ss")
    with open(data_path, "rb") as f: n_data = pickle.load(f)
    with open(feat_path, "rb") as f: f_data = pickle.load(f)
    return n_data, f_data

print("Loading hs0...")
n_data_hs0, f_data_hs0 = load_pickled_ss("SKIEUR_hs_0", SESSION_TYPE, dt)
print(f"  hs0: {len(n_data_hs0)} sessions")

print("Loading hs1...")
n_data_hs1, f_data_hs1 = load_pickled_ss("SKIEUR_hs_1", SESSION_TYPE, dt)
print(f"  hs1: {len(n_data_hs1)} sessions")

# Add Velocity_x
for f_df in f_data_hs0 + f_data_hs1:
    pos = f_df["Position"].values
    vel = np.diff(pos); vel = np.append(0, vel)
    vel = vel * 100; vel[~np.isfinite(vel)] = 0
    f_df["Velocity_x"] = vel

n_data_all_raw = list(n_data_hs0) + list(n_data_hs1)
f_data_all_raw = list(f_data_hs0) + list(f_data_hs1)
n_hs0 = len(n_data_hs0)

MIN_GC = 10
n_data_all, f_data_all = [], []
n_hs0_filtered = 0
for i, nd in enumerate(n_data_all_raw):
    if nd.shape[0] >= MIN_GC:
        n_data_all.append(nd)
        f_data_all.append(f_data_all_raw[i])
        if i < n_hs0: n_hs0_filtered += 1

n_hs0 = n_hs0_filtered
print(f"After gc>={MIN_GC}: {len(n_data_all)} sessions (hs0={n_hs0}, hs1={len(n_data_all)-n_hs0})")

example = f_data_all[0]
print(f"Example: {example.shape[0]:,} tp, {n_data_all[0].shape[0]} neurons")
print(f"  Velocity_x: [{example['Velocity_x'].min():.1f}, {example['Velocity_x'].max():.1f}]")
print(f"  Position:   [{example['Position'].min():.1f}, {example['Position'].max():.1f}]")

In [ ]:
# ============================================================
# Helper functions: macro-epoch + unified extraction + Lie Algebra
# ============================================================


def preprocess_data(data_list, method="l2"):
    """Preprocess list of (time, neurons) arrays.
    method='l2': per-timepoint L2 normalization (for cosine distance)
    method='zscore': per-neuron Z-score across time (for euclidean distance)
    """
    out = []
    for d in data_list:
        if method == "zscore":
            mean = np.mean(d, axis=0, keepdims=True)
            std = np.std(d, axis=0, keepdims=True)
            std[std == 0] = 1e-9
            out.append(((d - mean) / std).astype(np.float32))
        else:  # l2
            norms = np.linalg.norm(d, axis=1, keepdims=True)
            norms[norms == 0] = 1e-9
            out.append((d / norms).astype(np.float32))
    return out


def extract_macro_epochs(n_data_session, f_df, condition_val, dt,
                          min_duration=2.0, label_col="Velocity_x"):
    """Extract contiguous macro-epochs of the same Condition."""
    conditions = f_df["Condition"].values
    mask = (conditions == condition_val)
    min_bins = int(min_duration / dt)
    epochs_n, epochs_l = [], []
    in_epoch, start = False, 0
    for i in range(len(mask)):
        if mask[i] and not in_epoch:
            start = i; in_epoch = True
        elif not mask[i] and in_epoch:
            if i - start >= min_bins:
                epochs_n.append(n_data_session[:, start:i].T.astype(np.float32))
                epochs_l.append(f_df[label_col].values[start:i].astype(np.float32))
            in_epoch = False
    if in_epoch and (len(mask) - start) >= min_bins:
        epochs_n.append(n_data_session[:, start:].T.astype(np.float32))
        epochs_l.append(f_df[label_col].values[start:].astype(np.float32))
    return epochs_n, epochs_l, len(epochs_n)


def extract_epochs(n_data_session, f_df, condition_val, dt,
                   label_col="Velocity_x", step=1):
    """Unified extraction: macro-epochs or peri-event windows."""
    if USE_MACRO_EPOCH:
        epochs_n, epochs_l, n_ep = extract_macro_epochs(
            n_data_session, f_df, condition_val, dt,
            min_duration=MIN_EPOCH_DUR, label_col=label_col)
        if step > 1:
            epochs_n = [e[::step].astype(np.float32) for e in epochs_n]
            epochs_l = [l[::step].astype(np.float32) for l in epochs_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        epochs_n = preprocess_data(epochs_n, method=method)
        if TAU_SHIFT > 0:
            epochs_n, epochs_l = zip(*[(e_n[TAU_SHIFT:], e_l[:-TAU_SHIFT])
                                        for e_n, e_l in zip(epochs_n, epochs_l)])
            epochs_n, epochs_l = list(epochs_n), list(epochs_l)
        return epochs_n, epochs_l, n_ep
    else:
        trigger_mask = (f_df["Condition"].values == condition_val) & (f_df["Frequency_changes"].values == 1)
        trigger_indices = np.where(trigger_mask)[0]
        n_pre = int(t_pre / dt)
        n_post = int(t_post / dt)
        windows_n, windows_l = [], []
        for idx in trigger_indices:
            start = idx - n_pre
            end = idx + n_post + 1
            if start >= 0 and end <= n_data_session.shape[1]:
                windows_n.append(n_data_session[:, start:end].T.astype(np.float32))
                windows_l.append(f_df[label_col].values[start:end].astype(np.float32))
        if step > 1:
            windows_n = [w[::step].astype(np.float32) for w in windows_n]
            windows_l = [l[::step].astype(np.float32) for l in windows_l]
        method = "l2" if CEBRA_DISTANCE == "cosine" else "zscore"
        windows_n = preprocess_data(windows_n, method=method)
        if TAU_SHIFT > 0:
            windows_n, windows_l = zip(*[(w_n[TAU_SHIFT:], w_l[:-TAU_SHIFT])
                                          for w_n, w_l in zip(windows_n, windows_l)])
            windows_n, windows_l = list(windows_n), list(windows_l)
        return windows_n, windows_l, len(windows_n)


def _fit_lie_lstsq(r, x_dot, dt=0.005):
    """OLS + skew-symmetrization (original method)."""
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0)
    U_rot = r * x_dot[:, np.newaxis]
    U = np.hstack([U_rot, r])
    weights_T, _, _, _ = np.linalg.lstsq(U, dr_dt, rcond=None)
    weights = weights_T.T
    J_ols = weights[:, :N]
    J_skew = 0.5 * (J_ols - J_ols.T)
    norm_total = np.linalg.norm(J_ols)
    norm_skew = np.linalg.norm(J_skew)
    sr = norm_skew / norm_total if norm_total > 1e-9 else 0
    dR_pred = U_rot @ J_skew.T + r @ weights[:, N:].T
    ss_res = np.sum((dr_dt - dR_pred) ** 2)
    ss_tot = np.sum((dr_dt - np.mean(dr_dt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-9 else 0
    return J_skew, sr, r2, J_ols


def _fit_lie_pytorch(r, x_dot, dt=0.005, n_iter=500, lr=1e-3):
    """Constrained optimization: J_skew = W - W^T via PyTorch.
    Solves min_{S^T=-S} ||dR/dt - S R x - L R||^2 directly.
    """
    import torch
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0)
    R_t = torch.tensor(r, dtype=torch.float32)
    X_t = torch.tensor(x_dot, dtype=torch.float32).reshape(-1, 1)
    dR_t = torch.tensor(dr_dt, dtype=torch.float32)
    W = torch.zeros(N, N, requires_grad=True)
    L_t = torch.zeros(N, N, requires_grad=True)
    opt = torch.optim.Adam([W, L_t], lr=lr)
    for _ in range(n_iter):
        opt.zero_grad()
        J_t = W - W.T
        dR_pred = (R_t * X_t) @ J_t.T + R_t @ L_t.T
        loss = torch.mean((dR_t - dR_pred) ** 2)
        loss.backward()
        opt.step()
    with torch.no_grad():
        J_skew = (W - W.T).numpy()
        L_np = L_t.numpy()
    U = np.hstack([r * x_dot[:, None], r])
    w_T, _, _, _ = np.linalg.lstsq(U, dr_dt, rcond=None)
    J_ols = w_T.T[:, :N]
    sr = np.linalg.norm(J_skew) / np.linalg.norm(J_ols) if np.linalg.norm(J_ols) > 1e-9 else 0
    dR_pred = (r * x_dot[:, None]) @ J_skew.T + r @ L_np.T
    ss_res = np.sum((dr_dt - dR_pred) ** 2)
    ss_tot = np.sum((dr_dt - np.mean(dr_dt)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-9 else 0
    return J_skew, sr, r2, J_ols


def fit_lie_algebra_with_leak(r, x_dot, dt=0.005, n_iter=500, lr=1e-3):
    """Fit dR/dt = J_skew * R * x_dot + L * R.
    Method controlled by global LIE_METHOD.
    """
    if LIE_METHOD == "lstsq":
        return _fit_lie_lstsq(r, x_dot, dt)
    else:
        return _fit_lie_pytorch(r, x_dot, dt, n_iter, lr)

print("Helpers ready: extract_macro_epochs, extract_epochs, fit_lie_algebra_with_leak")

In [ ]:
# ============================================================
# Macro-Epoch / Peri-Event Extraction Report
# Runs extract_epochs() for all sessions and prints counts.
# Use this to check epoch/trigger counts before heavy computation.
# ============================================================
for label_col, driver_name in [("Velocity_x", "Velocity"), ("Position", "Position")]:
    print("=" * 50)
    print(f"  {driver_name}-Driven")
    print("=" * 50)
    for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
        total_epochs, total_pts, total_sec = 0, 0, 0
        for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
            epochs_n, epochs_l, n_ep = extract_epochs(
                n_data_session, f_df, val, dt, label_col=label_col)
            total_epochs += n_ep
            total_pts += sum(e.shape[0] for e in epochs_n)
            total_sec += sum(e.shape[0] for e in epochs_n) * dt
        mode = "macro-epoch" if USE_MACRO_EPOCH else "peri-event"
        print(f"  {cond_name}: {total_epochs} {mode}s, {total_pts:,} pts, {total_sec:.0f}s total")
    print()

print("Proceed to analysis cells.")

## 1. Lie Algebra Dynamics

Equation: $d\mathbf{R}/dt = \mathbf{J}_{\text{skew}} \cdot \mathbf{R} \cdot x + \mathbf{L} \cdot \mathbf{R}$

$\mathbf{J}_{\text{skew}}$ = rotation generator, $\mathbf{L}$ = leak/dissipation.

### 1.1 Velocity-Driven

In [ ]:
# ==========================================
# 2. Lie Algebra Generator Analysis
# ==========================================
# Uses fit_lie_algebra_with_leak from the helpers cell (cell 2).
# LIE_METHOD config controls OLS vs PyTorch (default: lstsq).

# Run on all Skieur sessions
all_results = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T  # (Time, Neurons)
    x_dot_full = f_df["Velocity_x"].values  # signed velocity
    conditions = f_df["Condition"].values
    hs_label = "hs0" if idx < n_hs0 else "hs1"

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Velocity_x")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_dot_sub = np.concatenate(epochs_l, axis=0)

        try:
            J_skew, skew_ratio, r2, J_full = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
            all_results.append({
                "Subject": "SKIEUR",
                "Session_Idx": idx,
                "Headstage": hs_label,
                "Condition": label,
                "Skewness_Ratio": skew_ratio,
                "R2": r2,
                "N_Neurons": r_sub.shape[1],
                "N_Timepoints": r_sub.shape[0],
            })
        except Exception as e:
            print(f"Error SKIEUR session {idx} [{label}]: {e}")

results_df = pd.DataFrame(all_results)
print()
print("Analysis complete: " + str(len(results_df)) + " entries")
print(results_df.groupby(["Headstage", "Condition"])[["Skewness_Ratio", "R2"]].mean().round(4))

In [ ]:
# ============================================================
# Velocity Lie Algebra — Visualization & Paired Statistics
# ============================================================
from scipy.stats import ttest_rel

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=results_df, x="Headstage", y="Skewness_Ratio", hue="Condition",
            ax=axes[0], palette="viridis", errorbar="se")
axes[0].set_title("SKIEUR: Skewness Ratio (Velocity-Driven)")

sns.barplot(data=results_df, x="Headstage", y="R2", hue="Condition",
            ax=axes[1], palette="magma", errorbar="se")
axes[1].set_title("SKIEUR: Model R" + chr(0x00B2))

pivot = results_df.pivot_table(
    values=["Skewness_Ratio", "R2"],
    index=["Subject", "Session_Idx", "Headstage"], columns="Condition"
).dropna()

t_skew, p_skew = ttest_rel(pivot["Skewness_Ratio"]["Tracking"],
                           pivot["Skewness_Ratio"]["Playback"])

tr_vals = pivot["Skewness_Ratio"]["Tracking"].values
pb_vals = pivot["Skewness_Ratio"]["Playback"].values
axes[2].scatter(tr_vals, pb_vals, alpha=0.6, c="steelblue")
lims = [min(tr_vals.min(), pb_vals.min()) - 0.02,
        max(tr_vals.max(), pb_vals.max()) + 0.02]
axes[2].plot(lims, lims, "--", color="gray", alpha=0.5, label="y = x")
axes[2].set_xlabel("Tracking Skewness Ratio"); axes[2].set_ylabel("Playback Skewness Ratio")
axes[2].set_title(f"Paired (N={len(pivot)})\nt={t_skew:.2f}, p={p_skew:.4f}")
axes[2].legend()
plt.suptitle("SKIEUR: Lie Algebra — Active Tracking vs Passive Playback (Velocity)",
             fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

print(f"Paired t-test (N={len(pivot)} pairs):")
print(f"  Skewness Ratio: t={t_skew:.3f}, p={p_skew:.6f}")
print(f"  Tracking  {tr_vals.mean():.4f} +/- {tr_vals.std():.4f}")
print(f"  Playback  {pb_vals.mean():.4f} +/- {pb_vals.std():.4f}")

In [ ]:
# ==========================================
# 4. Eigenvalue Stability Analysis
# ==========================================
min_points = 500
eig_results = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_dot_full = f_df["Velocity_x"].values
    conditions = f_df["Condition"].values
    hs_label = "hs0" if idx < n_hs0 else "hs1"

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Velocity_x")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_dot_sub = np.concatenate(epochs_l, axis=0)

        try:
            _, _, _, J_raw = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
            eigvals = np.linalg.eigvals(J_raw)
            real_mean = np.mean(np.abs(np.real(eigvals)))
            imag_mean = np.mean(np.abs(np.imag(eigvals)))
            eig_results.append({
                "Subject": "SKIEUR", "Session_Idx": idx, "Headstage": hs_label,
                "Condition": label, "Real_Mean": real_mean,
                "Imag_Mean": imag_mean,
                "Imag_Real_Ratio": imag_mean / real_mean if real_mean > 1e-9 else 0,
                "N_Neurons": r_full.shape[1],
            })
        except Exception as e:
            print(f"Eig error SKIEUR {idx} [{label}]: {e}")

eig_df = pd.DataFrame(eig_results)

pivot_eig = eig_df.pivot_table(
    values=["Real_Mean", "Imag_Mean"],
    index=["Subject", "Session_Idx", "Headstage"],
    columns="Condition"
).dropna()

t_real, p_real = ttest_rel(pivot_eig["Real_Mean"]["Tracking"], pivot_eig["Real_Mean"]["Playback"])
t_imag, p_imag = ttest_rel(pivot_eig["Imag_Mean"]["Tracking"], pivot_eig["Imag_Mean"]["Playback"])

print("Eigenvalue paired t-test (N=" + str(len(pivot_eig)) + "):")
print("  |Real| (Dissipation): Tracking=" + str(round(pivot_eig["Real_Mean"]["Tracking"].mean(), 4)) + ", Playback=" + str(round(pivot_eig["Real_Mean"]["Playback"].mean(), 4)) + ", t=" + str(round(t_real, 3)) + ", p=" + str(round(p_real, 6)))
print("  |Imag| (Rotation):    Tracking=" + str(round(pivot_eig["Imag_Mean"]["Tracking"].mean(), 4)) + ", Playback=" + str(round(pivot_eig["Imag_Mean"]["Playback"].mean(), 4)) + ", t=" + str(round(t_imag, 3)) + ", p=" + str(round(p_imag, 6)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
configs = [
    (axes[0], "Real_Mean", "Dissipation (|Real|)\n" + "p=" + str(round(p_real, 4)), p_real, "red"),
    (axes[1], "Imag_Mean", "Rotation (|Imag|)\n" + "p=" + str(round(p_imag, 4)), p_imag, "blue"),
]
for ax, col, ttl, p_val, color in configs:
    for _, row in pivot_eig.iterrows():
        ax.plot([0, 1], [row[col]["Tracking"], row[col]["Playback"]],
                "o-", color="gray", alpha=0.3, markersize=4, linewidth=0.8)
    ax.plot([0, 1], [pivot_eig[col]["Tracking"].mean(), pivot_eig[col]["Playback"].mean()],
            "o-", color=color, linewidth=3, markersize=10, label="Mean")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Tracking", "Playback"])
    ax.set_ylabel(col)
    ax.set_title(ttl)
    ax.legend()

plt.suptitle("SKIEUR: Dynamical Stability Signatures", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 1.2 Position-Driven

In [ ]:
# ==========================================
# 4.1 Position-Driven Lie Algebra
# ==========================================

# Reuse the same fit function, but drive with Position instead of Velocity
# dR/dt = J * R * x + L * R

pos_results = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_full = f_df["Position"].values  # position as driver
    conditions = f_df["Condition"].values
    hs_label = "hs0" if idx < n_hs0 else "hs1"

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Position")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_sub = np.concatenate(epochs_l, axis=0)

        try:
            J_skew, skew_ratio, r2, J_full = fit_lie_algebra_with_leak(r_sub, x_sub)
            pos_results.append({
                "Subject": "SKIEUR",
                "Session_Idx": idx,
                "Headstage": hs_label,
                "Condition": label,
                "Skewness_Ratio": skew_ratio,
                "R2": r2,
                "N_Neurons": r_sub.shape[1],
            })
        except Exception as e:
            print(f"Error SKIEUR {idx} [{label}]: {e}")

pos_df = pd.DataFrame(pos_results)
print("Position-driven analysis: " + str(len(pos_df)) + " entries")
print(pos_df.groupby(["Headstage", "Condition"])[["Skewness_Ratio", "R2"]].mean().round(4))

# ---- Comparison: Velocity vs Position ----
print()
print("=" * 65)
print("  Velocity-driven vs Position-driven Skewness Ratio")
print("=" * 65)
# Velocity results are in results_df
print()
print("  Driver       Condition     Skewness (mean)    R2 (mean)")
print("  " + "-" * 55)
for driver, df in [("Velocity", results_df), ("Position", pos_df)]:
    for cond in ["Tracking", "Playback"]:
        subset = df[df["Condition"] == cond]
        if len(subset) > 0:
            sk = subset["Skewness_Ratio"].mean()
            r2 = subset["R2"].mean()
            print("  {:<12s} {:<12s}  {:.4f}            {:.4f}".format(driver, cond, sk, r2))

# Paired t-test for position-driven
from scipy.stats import ttest_rel
pivot_pos = pos_df.pivot_table(
    values=["Skewness_Ratio", "R2"],
    index=["Subject", "Session_Idx", "Headstage"],
    columns="Condition"
).dropna()

t_pos, p_pos = ttest_rel(pivot_pos["Skewness_Ratio"]["Tracking"],
                          pivot_pos["Skewness_Ratio"]["Playback"])
print()
print("Position-driven paired t-test (N=" + str(len(pivot_pos)) + "):")
print("  t={:.3f}, p={:.6f}".format(t_pos, p_pos))
print("  Tracking  Skewness: {:.4f} +/- {:.4f}".format(pivot_pos["Skewness_Ratio"]["Tracking"].mean(), pivot_pos["Skewness_Ratio"]["Tracking"].std()))
print("  Playback  Skewness: {:.4f} +/- {:.4f}".format(pivot_pos["Skewness_Ratio"]["Playback"].mean(), pivot_pos["Skewness_Ratio"]["Playback"].std()))

# Barplot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=pos_df, x="Headstage", y="Skewness_Ratio", hue="Condition",
            ax=axes[0], palette="viridis", errorbar="se")
axes[0].set_title("SKIEUR: Position-Driven Skewness Ratio")
axes[0].set_ylabel("Skewness Ratio")

# Overlay velocity vs position tracking means
drivers = []
for driver, df in [("Velocity", results_df), ("Position", pos_df)]:
    for _, row in df.iterrows():
        drivers.append({"Driver": driver, "Condition": row["Condition"], "Skewness_Ratio": row["Skewness_Ratio"]})
driver_df = pd.DataFrame(drivers)
sns.barplot(data=driver_df, x="Driver", y="Skewness_Ratio", hue="Condition",
            ax=axes[1], palette="viridis", errorbar="se")
axes[1].set_title("Velocity vs Position Driver")
axes[1].set_ylabel("Skewness Ratio")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Position Lie Algebra — Visualization & Paired Statistics
# ============================================================
from scipy.stats import ttest_rel

pivot_pos = pos_df.pivot_table(
    values=["Skewness_Ratio", "R2"],
    index=["Subject", "Session_Idx", "Headstage"], columns="Condition"
).dropna()

t_pos, p_pos = ttest_rel(pivot_pos["Skewness_Ratio"]["Tracking"],
                          pivot_pos["Skewness_Ratio"]["Playback"])

print(f"Position-driven paired t-test (N={len(pivot_pos)}):")
print(f"  Skewness: t={t_pos:.3f}, p={p_pos:.6f}")
print(f"  Tracking  {pivot_pos['Skewness_Ratio']['Tracking'].mean():.4f} +/- {pivot_pos['Skewness_Ratio']['Tracking'].std():.4f}")
print(f"  Playback  {pivot_pos['Skewness_Ratio']['Playback'].mean():.4f} +/- {pivot_pos['Skewness_Ratio']['Playback'].std():.4f}")

# Comparison barplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=pos_df, x="Headstage", y="Skewness_Ratio", hue="Condition",
            ax=axes[0], palette="viridis", errorbar="se")
axes[0].set_title("Position-Driven Skewness Ratio")

drivers = []
for driver, df in [("Velocity", results_df), ("Position", pos_df)]:
    for _, row in df.iterrows():
        drivers.append({"Driver": driver, "Condition": row["Condition"],
                        "Skewness_Ratio": row["Skewness_Ratio"]})
driver_df = pd.DataFrame(drivers)
sns.barplot(data=driver_df, x="Driver", y="Skewness_Ratio", hue="Condition",
            ax=axes[1], palette="viridis", errorbar="se")
axes[1].set_title("Velocity vs Position Driver")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Position Eigenvalue Stability Analysis
# ============================================================
min_points = 500
eig_results_pos = []

for idx, (n_data_session, f_df) in enumerate(zip(n_data_all, f_data_all)):
    r_full = n_data_session.T
    x_full = f_df["Position"].values
    conditions = f_df["Condition"].values
    hs_label = "hs0" if idx < n_hs0 else "hs1"

    for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
        # Extract peri-event windows
        epochs_n, epochs_l, n_ep = extract_epochs(
            n_data_session, f_df, val, dt,
            label_col="Position")
        if n_ep < 1:
            continue
        r_sub = np.concatenate(epochs_n, axis=0)
        x_sub = np.concatenate(epochs_l, axis=0)

        try:
            _, _, _, J_raw = fit_lie_algebra_with_leak(r_sub, x_sub)
            eigvals = np.linalg.eigvals(J_raw)
            real_mean = np.mean(np.abs(np.real(eigvals)))
            imag_mean = np.mean(np.abs(np.imag(eigvals)))
            eig_results_pos.append({
                "Subject": "SKIEUR", "Session_Idx": idx, "Headstage": hs_label,
                "Condition": label,
                "Real_Mean": real_mean, "Imag_Mean": imag_mean,
                "Imag_Real_Ratio": imag_mean / real_mean if real_mean > 1e-9 else 0,
                "N_Neurons": r_full.shape[1],
            })
        except Exception as e:
            print(f"Eig error SKIEUR {idx} [{label}]: {e}")

eig_pos_df = pd.DataFrame(eig_results_pos)

pivot_eig_pos = eig_pos_df.pivot_table(
    values=["Real_Mean", "Imag_Mean"],
    index=["Subject", "Session_Idx", "Headstage"], columns="Condition"
).dropna()

t_real_p, p_real_p = ttest_rel(pivot_eig_pos["Real_Mean"]["Tracking"],
                                pivot_eig_pos["Real_Mean"]["Playback"])
t_imag_p, p_imag_p = ttest_rel(pivot_eig_pos["Imag_Mean"]["Tracking"],
                                pivot_eig_pos["Imag_Mean"]["Playback"])

print("Position Eigenvalue paired t-test (N=" + str(len(pivot_eig_pos)) + "):")
print(f"  |Real| (Dissipation): Tracking={pivot_eig_pos['Real_Mean']['Tracking'].mean():.4f}, "
      f"Playback={pivot_eig_pos['Real_Mean']['Playback'].mean():.4f}, "
      f"t={t_real_p:.3f}, p={p_real_p:.6f}")
print(f"  |Imag| (Rotation):    Tracking={pivot_eig_pos['Imag_Mean']['Tracking'].mean():.4f}, "
      f"Playback={pivot_eig_pos['Imag_Mean']['Playback'].mean():.4f}, "
      f"t={t_imag_p:.3f}, p={p_imag_p:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
configs = [
    (axes[0], "Real_Mean", "Dissipation (|Real|)\np=" + str(round(p_real_p, 4)), "red"),
    (axes[1], "Imag_Mean", "Rotation (|Imag|)\np=" + str(round(p_imag_p, 4)), "blue"),
]
for ax, col, ttl, color in configs:
    for _, row in pivot_eig_pos.iterrows():
        ax.plot([0, 1], [row[col]["Tracking"], row[col]["Playback"]],
                "o-", color="gray", alpha=0.3, markersize=4, linewidth=0.8)
    ax.plot([0, 1], [pivot_eig_pos[col]["Tracking"].mean(), pivot_eig_pos[col]["Playback"].mean()],
            "o-", color=color, linewidth=3, markersize=10, label="Mean")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Tracking", "Playback"])
    ax.set_ylabel(col); ax.set_title(ttl); ax.legend()
plt.suptitle("SKIEUR: Dynamical Stability — Position-Driven", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## 2. CEBRA Manifold Analysis

3D manifold visualization + statistical validation (InfoNCE loss, k-NN decoding R$^2$, shuffle test).

**Note:** May consume significant GPU memory. `gc.collect()` + `torch.cuda.empty_cache()` are called after each section.

### 2.1 Velocity-Driven

In [ ]:
# ==========================================
# 5. 3D Manifold HTML Visualization (Plotly)
# ==========================================
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
    import cebra
    from cebra import CEBRA
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_CEBRA = True
    print("CEBRA is available")
except ImportError:
    HAS_CEBRA = False
    print("CEBRA not installed. This cell requires: pip install cebra")

if HAS_CEBRA:

    def run_cebra_skieur(n_data_list, f_data_list, n_hs0, dt=0.005):
        """Train CEBRA on L2-normalized neural data, split by Tracking/Playback.
        Uses signed Velocity_x as labels.
        """
        results = {}
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            neural_list = []
            label_list = []
            sid_list = []

            for s_idx, (n_raw, f_df) in enumerate(zip(n_data_list, f_data_list)):
                # Extract peri-event windows (±0.3s around Frequency_changes==1)
                epochs_n, epochs_l, n_ep = extract_epochs(
                    n_raw, f_df, val, dt,
                    label_col="Velocity_x")
                if n_ep < 1:
                    continue

                # Concatenate macro-epochs within this session.
                # Condition switches form natural boundaries between epochs.
                curr_n = np.concatenate(epochs_n, axis=0)
                curr_l = np.concatenate(epochs_l, axis=0).reshape(-1, 1)

                neural_list.append(curr_n)
                label_list.append(curr_l)
                sid_list.append(np.full(len(curr_l), s_idx))

            if not neural_list:
                continue

            hs_counts = sum(1 for i in range(len(neural_list)) if i < n_hs0)
            print(f"  {label}: {len(neural_list)} sessions (hs0={hs_counts}, hs1={len(neural_list)-hs_counts})")

            model = CEBRA(
                model_architecture=CEBRA_ARCH,
                output_dimension=3,
                max_iterations=3000,
                batch_size=512,
                learning_rate=3e-4,
                temperature=1.5,
                distance=CEBRA_DISTANCE,
                conditional="time_delta",
                device="cuda",
                verbose=True
            )
            model.fit(neural_list, label_list)

            embeddings_list = []
            for i, curr_n_norm in enumerate(neural_list):
                curr_emb = model.transform(curr_n_norm, session_id=i)
                embeddings_list.append(curr_emb)

            emb_all = np.concatenate(embeddings_list, axis=0)
            results[label] = (
                emb_all,
                np.concatenate(label_list).flatten(),
                np.concatenate(sid_list)
            )
            print(f"  {label} complete.")

        return results

    def visualize_skieur_webgl(cebra_res, save_dir, downsample_factor=10):
        """Plotly WebGL 3D scatter: Tracking (left) vs Playback (right).
        Optimized: downsampling + float rounding + CDN for small HTML."""
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=("<b>Active Tracking (SKIEUR)</b>",
                            "<b>Passive Playback (SKIEUR)</b>")
        )

        all_embs = np.concatenate([res[0] for res in cebra_res.values()])
        axis_max = np.max(np.abs(all_embs)) * 1.1

        all_labels = np.concatenate([res[1] for res in cebra_res.values()])
        cmin = float(np.percentile(all_labels, 2))
        cmax = float(np.percentile(all_labels, 98))

        fig.add_trace(
            go.Scatter3d(
                x=[0, 0], y=[0, 0], z=[0, 0],
                mode="markers",
                marker=dict(
                    size=0.001,
                    color=[cmin, cmax],
                    colorscale="Plasma",
                    cmin=cmin, cmax=cmax,
                    showscale=True,
                    colorbar=dict(title="Velocity (signed)", x=1.05, thickness=15),
                    line=dict(width=0)
                ),
                showlegend=False, hoverinfo="skip"
            ),
            row=1, col=2
        )

        for i, cond in enumerate(["Tracking", "Playback"]):
            if cond not in cebra_res:
                continue
            emb, labels, sess_ids = cebra_res[cond]

            # In-plot downsampling for HTML size reduction
            if downsample_factor > 1:
                emb = emb[::downsample_factor]
                labels = labels[::downsample_factor]
                sess_ids = sess_ids[::downsample_factor]

            # Float precision reduction
            emb = np.round(emb, 4)
            labels_rounded = np.round(labels, 2)

            unique_sessions = np.unique(sess_ids)

            for s_id in unique_sessions:
                mask = (sess_ids == s_id)
                if not np.any(mask):
                    continue

                dynamic_size = 3 + 8 * np.abs(labels[mask]) / max(abs(cmin), abs(cmax), 1e-9)
                dynamic_size = np.round(dynamic_size, 1)

                fig.add_trace(
                    go.Scatter3d(
                        x=emb[mask, 0], y=emb[mask, 1], z=emb[mask, 2],
                        mode="markers",
                        marker=dict(
                            size=dynamic_size,
                            line=dict(width=0),
                            color=labels_rounded[mask],
                            colorscale="Plasma",
                            cmin=cmin, cmax=cmax,
                            opacity=0.4,
                            showscale=False
                        ),
                        name=f"Sess {int(s_id)}",
                        legendgroup=f"Sess {int(s_id)}",
                        showlegend=bool(i == 0)
                    ),
                    row=1, col=i + 1
                )

        scene_config = dict(
            xaxis=dict(range=[-axis_max, axis_max], visible=False),
            yaxis=dict(range=[-axis_max, axis_max], visible=False),
            zaxis=dict(range=[-axis_max, axis_max], visible=False),
            aspectmode="cube"
        )

        fig.update_layout(
            height=700,
            title_text="Shared Neural Manifold Topology: SKIEUR",
            title_x=0.5,
            scene1=scene_config,
            scene2=scene_config,
            margin=dict(l=0, r=0, b=0, t=60),
        template="plotly_white",
            legend=dict(title="Sessions", x=1.1, y=0.5)
        )

        file_name = f"CEBRA_Velocity_{CEBRA_DISTANCE}_{'macro' if USE_MACRO_EPOCH else 'peri'}.html"
        save_path = os.path.join(save_dir, file_name)
        fig.write_html(save_path, include_plotlyjs='cdn')
        print(f"Saved: {save_path}")

    # ---- Sample 5 sessions chronologically, then execute CEBRA ----
    MIN_GC = 10
    N_SAMPLE = 5
    valid_idx = [i for i, nd in enumerate(n_data_all) if nd.shape[0] >= MIN_GC]

    if len(valid_idx) <= N_SAMPLE:
        chosen = valid_idx
    else:
        chosen = [valid_idx[int(round(i * (len(valid_idx) - 1) / (N_SAMPLE - 1)))] for i in range(N_SAMPLE)]
        chosen = list(dict.fromkeys(chosen))

    n_subset = [n_data_all[i] for i in chosen]
    f_subset = [f_data_all[i] for i in chosen]
    hs0_subset = sum(1 for i in chosen if i < n_hs0)

    print(f"Sampled {len(chosen)} / {len(valid_idx)} sessions (gc >= {MIN_GC}):")
    for j, i in enumerate(chosen):
        print(f"  {j+1}. session {i}: {n_data_all[i].shape[0]} neurons, {n_data_all[i].shape[1]} timepoints")

    save_directory = r"C:\\Users\\PenPen\\Desktop\\Ferret\\Results&PLots\\Web"
    if not os.path.exists(save_directory):
        save_directory = os.path.join(os.getcwd(), "output")
        os.makedirs(save_directory, exist_ok=True)
        print(f"Fallback: {save_directory}")

    print("=" * 60)
    print("SKIEUR 3D Manifold - signed Velocity_x, 5-session sample")
    print("Output: " + save_directory)
    print("=" * 60)

    cebra_results = run_cebra_skieur(n_subset, f_subset, hs0_subset)
    visualize_skieur_webgl(cebra_results, save_directory)

    print("Done! Open CEBRA_Manifold_SKIEUR_updated.html in a browser.")
else:
    print("Skipped: CEBRA not installed. Install with: pip install cebra")

In [ ]:
# --- GPU cleanup ---
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

In [ ]:
# ============================================================
# 5. CEBRA Manifold Statistical Validation -- Isolated 5-Fold CV
#
#    Tracking and Playback are FULLY isolated:
#    each condition gets its own CEBRA model + 5-fold CV.
#    R2, loss curves, and scatter plots per condition.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from cebra import CEBRA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

K = 5
STEP = 5

n_raw = n_data_all[-1]
f_df = f_data_all[-1]
print(f"Session {len(n_data_all)-1} (last): {n_raw.shape[0]} neurons")

# ---- 1. Extract + chunk epochs per condition ----
cond_data = {}  # condition -> {n: [chunks], l: [chunks]}

for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
    en, el, n_ep = extract_epochs(n_raw, f_df, val, dt, label_col="Velocity_x", step=STEP)
    chunks_n, chunks_l = [], []
    for e_n, e_l in zip(en, el):
        n = e_n.shape[0]
        chunk_len = n // K
        for k in range(K):
            start = k * chunk_len
            end = n if k == K - 1 else (k + 1) * chunk_len
            chunks_n.append(e_n[start:end].astype(np.float32))
            chunks_l.append(e_l[start:end].astype(np.float32))

    chunks_n = preprocess_data(chunks_n,
        method="l2" if CEBRA_DISTANCE == "cosine" else "zscore")
    cond_data[cond_name] = {"n": chunks_n, "l": chunks_l}
    print(f"  {cond_name}: {len(chunks_n)} chunks, ~{chunks_n[0].shape[0]} pts/chunk")

# ---- 2. Isolated K-fold CV per condition ----
all_results = {}

for cond_name in ["Tracking", "Playback"]:
    chunks_n = cond_data[cond_name]["n"]
    chunks_l = cond_data[cond_name]["l"]

    r2_true_folds, r2_shuf_folds = [], []
    loss_true_folds, loss_shuf_folds = [], []

    print()
    print("=" * 60)
    print(f"  {cond_name}: {K}-fold CV (fully isolated)")
    print("=" * 60)

    fig_loss, axes_loss = plt.subplots(1, K, figsize=(4*K, 4), sharey=True)
    fig_scatter, axes_scatter = plt.subplots(2, K, figsize=(4*K, 8))

    for fold in range(K):
        test_idx = [fold]
        train_idx = [i for i in range(K) if i != fold]

        train_n = [chunks_n[i] for i in train_idx]
        train_l = [chunks_l[i] for i in train_idx]
        train_l_shuf = [np.random.permutation(l) for l in train_l]  # random permutation destroys time structure

        X_test = chunks_n[fold]
        y_test = chunks_l[fold]

        train_pts = sum(l.shape[0] for l in train_l)
        print(f"\n  Fold {fold+1}/{K}: train={train_pts:,} ({len(train_n)} chunks), test={X_test.shape[0]:,}")

        cebra_true = CEBRA(
            model_architecture=CEBRA_ARCH, output_dimension=3,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True
        )
        cebra_true.fit(train_n, train_l)

        cebra_shuffle = CEBRA(
            model_architecture=CEBRA_ARCH, output_dimension=3,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True
        )
        cebra_shuffle.fit(train_n, train_l_shuf)

        # Loss curves
        ax_l = axes_loss[fold]
        ax_l.plot(cebra_true.state_dict_['loss'], color='darkred', label='True')
        ax_l.plot(cebra_shuffle.state_dict_['loss'], color='gray', ls='--', label='Shuffle')
        ax_l.set_title(f"Fold {fold+1}"); ax_l.set_xlabel("Iter"); ax_l.legend(fontsize=6)
        loss_true_folds.append(cebra_true.state_dict_['loss'][-1])
        loss_shuf_folds.append(cebra_shuffle.state_dict_['loss'][-1])

        # k-NN decode
        emb_train = np.concatenate([cebra_true.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test = cebra_true.transform(X_test, session_id=0)
        emb_train_shuf = np.concatenate([cebra_shuffle.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test_shuf = cebra_shuffle.transform(X_test, session_id=0)

        y_train_flat = np.concatenate(train_l)
        y_train_shuf_flat = np.concatenate(train_l_shuf)

        decoder_true = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_shuffle = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_true.fit(emb_train, y_train_flat)
        decoder_shuffle.fit(emb_train_shuf, y_train_shuf_flat)

        y_pred_true = decoder_true.predict(emb_test)
        y_pred_shuf = decoder_shuffle.predict(emb_test_shuf)

        r2_t = r2_score(y_test, y_pred_true)
        r2_s = r2_score(y_test, y_pred_shuf)
        r2_true_folds.append(r2_t)
        r2_shuf_folds.append(r2_s)
        print(f"    True R2={r2_t:.4f}  Shuffle R2={r2_s:.4f}")

        # Scatter
        for row, yp, label in [(0, y_pred_true, "True"), (1, y_pred_shuf, "Shuffle")]:
            ax = axes_scatter[row, fold]
            ax.scatter(y_test[::5], yp[::5], alpha=0.3, s=1, c='darkred' if row==0 else 'gray')
            l = [y_test.min(), y_test.max()]
            ax.plot(l, l, '--', color='gray', alpha=0.5)
            ax.set_xlabel("True"); ax.set_ylabel("Pred")
            ax.set_title(f"{label} Fold {fold+1}")

    plt.suptitle(f"SKIEUR: {cond_name} - 5-Fold CV Loss Curves (Velocity)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
    plt.suptitle(f"SKIEUR: {cond_name} - 5-Fold CV True vs Predicted (Velocity)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

    all_results[cond_name] = {
        "r2_true": (np.mean(r2_true_folds), np.std(r2_true_folds)),
        "r2_shuf": (np.mean(r2_shuf_folds), np.std(r2_shuf_folds)),
        "loss_true": (np.mean(loss_true_folds), np.std(loss_true_folds)),
        "loss_shuf": (np.mean(loss_shuf_folds), np.std(loss_shuf_folds)),
        "r2_true_all": r2_true_folds,
        "r2_shuf_all": r2_shuf_folds,
    }

# ---- 3. Report ----
print()
print("=" * 60)
print("  Isolated 5-Fold CV CEBRA Validation Report (Velocity)")
print("=" * 60)
for cond_name in ["Tracking", "Playback"]:
    r = all_results[cond_name]
    print(f"\n  --- {cond_name} ---")
    print(f"  True  R2:  {r['r2_true'][0]:.4f} +/- {r['r2_true'][1]:.4f}")
    print(f"  Shuf  R2:  {r['r2_shuf'][0]:.4f} +/- {r['r2_shuf'][1]:.4f}")
    print(f"  True  loss: {r['loss_true'][0]:.4f} +/- {r['loss_true'][1]:.4f}")
    print(f"  Shuf  loss: {r['loss_shuf'][0]:.4f} +/- {r['loss_shuf'][1]:.4f}")
    if r['r2_true'][0] > 0.3 and r['r2_true'][0] > (r['r2_shuf'][0] + 0.2):
        print(f"  => PASSED")
    else:
        print(f"  => FAILED")

# Summary barplot
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
bars = [all_results["Tracking"]["r2_true"][0], all_results["Playback"]["r2_true"][0],
        all_results["Tracking"]["r2_shuf"][0], all_results["Playback"]["r2_shuf"][0]]
errs = [all_results["Tracking"]["r2_true"][1], all_results["Playback"]["r2_true"][1],
        all_results["Tracking"]["r2_shuf"][1], all_results["Playback"]["r2_shuf"][1]]
colors = ['darkred', 'darkblue', 'gray', 'lightgray']
ax.bar(x, bars, yerr=errs, color=colors, capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(['True Tracking', 'True Playback', 'Shuffle Tracking', 'Shuffle Playback'])
ax.set_ylabel("R2 (mean +/- std)")
ax.set_title("SKIEUR: Isolated 5-Fold CV R2 Summary (Velocity)")
plt.tight_layout(); plt.show()

print()
print(">>> Isolated CV complete.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
del cebra_true, cebra_shuffle
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

### 2.2 Position-Driven

In [ ]:
# ============================================================
# Position-Driven CEBRA 3D Manifold
# ============================================================
# NOTE: run_cebra_position now uses extract_epochs output directly
# (preprocessing is handled by extract_epochs, consistent with Velocity path).
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
    import cebra
    from cebra import CEBRA
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_CEBRA = True
    print("CEBRA is available")
except ImportError:
    HAS_CEBRA = False
    print("CEBRA not installed")

if HAS_CEBRA:

    def run_cebra_position(n_data_list, f_data_list, n_hs0):
        """Train CEBRA with Position as label.
        Preprocessing (z-score or L2) is delegated to extract_epochs;
        no extra normalization is applied here.
        """
        results = {}
        for val, label in [(0.0, "Tracking"), (1.0, "Playback")]:
            neural_list, label_list, sid_list = [], [], []
            for s_idx, (n_raw, f_df) in enumerate(zip(n_data_list, f_data_list)):
                epochs_n, epochs_l, n_ep = extract_epochs(
                    n_raw, f_df, val, dt, label_col="Position")
                if n_ep < 1:
                    continue
                curr_n = np.concatenate(epochs_n, axis=0)
                curr_l = np.concatenate(epochs_l, axis=0).reshape(-1, 1)
                neural_list.append(curr_n)
                label_list.append(curr_l)
                sid_list.append(np.full(len(curr_l), s_idx))
            if not neural_list:
                print(f"  {label} has no valid epochs.")
                continue
            model = CEBRA(
                model_architecture=CEBRA_ARCH, output_dimension=3,
                max_iterations=3000, batch_size=512, learning_rate=3e-4,
                temperature=1.5, distance=CEBRA_DISTANCE,
                conditional="time_delta", device="cuda", verbose=True
            )
            model.fit(neural_list, label_list)
            embeddings_list = []
            for i, curr_n in enumerate(neural_list):
                curr_emb = model.transform(curr_n, session_id=i)
                embeddings_list.append(curr_emb)
            emb_all = np.concatenate(embeddings_list, axis=0)
            results[label] = (
                emb_all,
                np.concatenate(label_list).flatten(),
                np.concatenate(sid_list)
            )
            print(f"  {label} complete.")
        return results


    def visualize_position_webgl(cebra_res, save_dir, downsample_factor=10):
        """Plotly WebGL 3D with Position coloring. Optimized for small HTML."""
        fig = make_subplots(
            rows=1, cols=2,
            specs=[[{"type": "scene"}, {"type": "scene"}]],
            subplot_titles=("<b>Active Tracking</b>", "<b>Passive Playback</b>")
        )
        all_embs = np.concatenate([res[0] for res in cebra_res.values()])
        axis_max = np.max(np.abs(all_embs)) * 1.1
        all_labels = np.concatenate([res[1] for res in cebra_res.values()])
        cmin = float(np.percentile(all_labels, 2))
        cmax = float(np.percentile(all_labels, 98))
        fig.add_trace(
            go.Scatter3d(x=[0, 0], y=[0, 0], z=[0, 0], mode="markers",
                marker=dict(size=0.001, color=[cmin, cmax], colorscale="Plasma",
                    cmin=cmin, cmax=cmax, showscale=True,
                    colorbar=dict(title="Position (cm)", x=1.05, thickness=15),
                    line=dict(width=0)),
                showlegend=False, hoverinfo="skip"),
            row=1, col=2
        )
        for i, cond in enumerate(["Tracking", "Playback"]):
            if cond not in cebra_res:
                continue
            emb, labels, sess_ids = cebra_res[cond]
            if downsample_factor > 1:
                emb = emb[::downsample_factor]
                labels = labels[::downsample_factor]
                sess_ids = sess_ids[::downsample_factor]
            emb = np.round(emb, 4)
            labels_rounded = np.round(labels, 2)
            unique_sessions = np.unique(sess_ids)
            for s_id in unique_sessions:
                mask = (sess_ids == s_id)
                if not np.any(mask):
                    continue
                dynamic_size = 3 + 6 * np.abs(labels[mask] - np.median(labels)) / max(abs(cmin), abs(cmax), 1e-9)
                dynamic_size = np.round(dynamic_size, 1)
                fig.add_trace(
                    go.Scatter3d(x=emb[mask, 0], y=emb[mask, 1], z=emb[mask, 2],
                        mode="markers",
                        marker=dict(size=dynamic_size, color=labels_rounded[mask],
                            colorscale="Plasma", cmin=cmin, cmax=cmax,
                            opacity=0.25, line=dict(width=0), showscale=False),
                        name=f"Sess {int(s_id)}", legendgroup=f"Sess {int(s_id)}",
                        showlegend=bool(i == 0)),
                    row=1, col=i + 1
                )
        scene_config = dict(
            xaxis=dict(range=[-axis_max, axis_max], visible=False),
            yaxis=dict(range=[-axis_max, axis_max], visible=False),
            zaxis=dict(range=[-axis_max, axis_max], visible=False),
            aspectmode="cube"
        )
        fig.update_layout(
            height=700, title_text="Neural Manifold (Position-Driven): SKIEUR",
            title_x=0.5, scene1=scene_config, scene2=scene_config,
            margin=dict(l=0, r=0, b=0, t=60),
            template="plotly_white",
            legend=dict(title="Sessions", x=1.1, y=0.5)
        )
        file_name = f"CEBRA_Position_{CEBRA_DISTANCE}_{'macro' if USE_MACRO_EPOCH else 'peri'}.html"
        save_path = os.path.join(save_dir, file_name)
        fig.write_html(save_path, include_plotlyjs='cdn')
        print(f"Saved: {save_path}")


    # ---- Sample 5 sessions chronologically, then execute CEBRA ----
    MIN_GC = 10
    N_SAMPLE = 5
    valid_idx = [i for i, nd in enumerate(n_data_all) if nd.shape[0] >= MIN_GC]
    if len(valid_idx) <= N_SAMPLE:
        chosen = valid_idx
    else:
        chosen = [valid_idx[int(round(i * (len(valid_idx) - 1) / (N_SAMPLE - 1)))] for i in range(N_SAMPLE)]
        chosen = list(dict.fromkeys(chosen))
    n_subset = [n_data_all[i] for i in chosen]
    f_subset = [f_data_all[i] for i in chosen]
    hs0_subset = sum(1 for i in chosen if i < n_hs0)
    print(f"Sampled {len(chosen)} / {len(valid_idx)} sessions")
    for j, i in enumerate(chosen):
        print(f"  {j+1}. session {i}: {n_data_all[i].shape[0]} neurons")
    save_directory = r"C:\Users\PenPen\Desktop\Ferret\Results&PLots\Web"
    if not os.path.exists(save_directory):
        save_directory = os.path.join(os.getcwd(), "output")
        os.makedirs(save_directory, exist_ok=True)
        print(f"Fallback: {save_directory}")
    print("=" * 60)
    print("SKIEUR 3D Manifold - Position-Driven")
    print("=" * 60)
    cebra_pos_results = run_cebra_position(n_subset, f_subset, hs0_subset)
    visualize_position_webgl(cebra_pos_results, save_directory)
    print("Done! Open CEBRA_Manifold_Position_SKIEUR.html")

else:
    print("Skipped: CEBRA not installed.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")

In [ ]:
# ============================================================
# 5. CEBRA Manifold Statistical Validation -- Isolated 5-Fold CV
#
#    Tracking and Playback are FULLY isolated:
#    each condition gets its own CEBRA model + 5-fold CV.
#    R2, loss curves, and scatter plots per condition.
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from cebra import CEBRA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

K = 5
STEP = 5

n_raw = n_data_all[-1]
f_df = f_data_all[-1]
print(f"Session {len(n_data_all)-1} (last): {n_raw.shape[0]} neurons")

# ---- 1. Extract + chunk epochs per condition ----
cond_data = {}  # condition -> {n: [chunks], l: [chunks]}

for val, cond_name in [(0.0, "Tracking"), (1.0, "Playback")]:
    en, el, n_ep = extract_epochs(n_raw, f_df, val, dt, label_col="Position", step=STEP)
    chunks_n, chunks_l = [], []
    for e_n, e_l in zip(en, el):
        n = e_n.shape[0]
        chunk_len = n // K
        for k in range(K):
            start = k * chunk_len
            end = n if k == K - 1 else (k + 1) * chunk_len
            chunks_n.append(e_n[start:end].astype(np.float32))
            chunks_l.append(e_l[start:end].astype(np.float32))

    chunks_n = preprocess_data(chunks_n,
        method="l2" if CEBRA_DISTANCE == "cosine" else "zscore")
    cond_data[cond_name] = {"n": chunks_n, "l": chunks_l}
    print(f"  {cond_name}: {len(chunks_n)} chunks, ~{chunks_n[0].shape[0]} pts/chunk")

# ---- 2. Isolated K-fold CV per condition ----
all_results = {}

for cond_name in ["Tracking", "Playback"]:
    chunks_n = cond_data[cond_name]["n"]
    chunks_l = cond_data[cond_name]["l"]

    r2_true_folds, r2_shuf_folds = [], []
    loss_true_folds, loss_shuf_folds = [], []

    print()
    print("=" * 60)
    print(f"  {cond_name}: {K}-fold CV (fully isolated)")
    print("=" * 60)

    fig_loss, axes_loss = plt.subplots(1, K, figsize=(4*K, 4), sharey=True)
    fig_scatter, axes_scatter = plt.subplots(2, K, figsize=(4*K, 8))

    for fold in range(K):
        test_idx = [fold]
        train_idx = [i for i in range(K) if i != fold]

        train_n = [chunks_n[i] for i in train_idx]
        train_l = [chunks_l[i] for i in train_idx]
        train_l_shuf = [np.random.permutation(l) for l in train_l]  # random permutation destroys time structure

        X_test = chunks_n[fold]
        y_test = chunks_l[fold]

        train_pts = sum(l.shape[0] for l in train_l)
        print(f"\n  Fold {fold+1}/{K}: train={train_pts:,} ({len(train_n)} chunks), test={X_test.shape[0]:,}")

        cebra_true = CEBRA(
            model_architecture=CEBRA_ARCH, output_dimension=3,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True
        )
        cebra_true.fit(train_n, train_l)

        cebra_shuffle = CEBRA(
            model_architecture=CEBRA_ARCH, output_dimension=3,
            max_iterations=3000, batch_size=2048, distance=CEBRA_DISTANCE,
            conditional="time_delta", device="cuda", verbose=True
        )
        cebra_shuffle.fit(train_n, train_l_shuf)

        # Loss curves
        ax_l = axes_loss[fold]
        ax_l.plot(cebra_true.state_dict_['loss'], color='darkred', label='True')
        ax_l.plot(cebra_shuffle.state_dict_['loss'], color='gray', ls='--', label='Shuffle')
        ax_l.set_title(f"Fold {fold+1}"); ax_l.set_xlabel("Iter"); ax_l.legend(fontsize=6)
        loss_true_folds.append(cebra_true.state_dict_['loss'][-1])
        loss_shuf_folds.append(cebra_shuffle.state_dict_['loss'][-1])

        # k-NN decode
        emb_train = np.concatenate([cebra_true.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test = cebra_true.transform(X_test, session_id=0)
        emb_train_shuf = np.concatenate([cebra_shuffle.transform(train_n[i], session_id=0) for i in range(len(train_n))], axis=0)
        emb_test_shuf = cebra_shuffle.transform(X_test, session_id=0)

        y_train_flat = np.concatenate(train_l)
        y_train_shuf_flat = np.concatenate(train_l_shuf)

        decoder_true = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_shuffle = KNeighborsRegressor(n_neighbors=5, weights='distance')
        decoder_true.fit(emb_train, y_train_flat)
        decoder_shuffle.fit(emb_train_shuf, y_train_shuf_flat)

        y_pred_true = decoder_true.predict(emb_test)
        y_pred_shuf = decoder_shuffle.predict(emb_test_shuf)

        r2_t = r2_score(y_test, y_pred_true)
        r2_s = r2_score(y_test, y_pred_shuf)
        r2_true_folds.append(r2_t)
        r2_shuf_folds.append(r2_s)
        print(f"    True R2={r2_t:.4f}  Shuffle R2={r2_s:.4f}")

        # Scatter
        for row, yp, label in [(0, y_pred_true, "True"), (1, y_pred_shuf, "Shuffle")]:
            ax = axes_scatter[row, fold]
            ax.scatter(y_test[::5], yp[::5], alpha=0.3, s=1, c='darkred' if row==0 else 'gray')
            l = [y_test.min(), y_test.max()]
            ax.plot(l, l, '--', color='gray', alpha=0.5)
            ax.set_xlabel("True"); ax.set_ylabel("Pred")
            ax.set_title(f"{label} Fold {fold+1}")

    plt.suptitle(f"SKIEUR: {cond_name} - 5-Fold CV Loss Curves (Position)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()
    plt.suptitle(f"SKIEUR: {cond_name} - 5-Fold CV True vs Predicted (Position)", fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

    all_results[cond_name] = {
        "r2_true": (np.mean(r2_true_folds), np.std(r2_true_folds)),
        "r2_shuf": (np.mean(r2_shuf_folds), np.std(r2_shuf_folds)),
        "loss_true": (np.mean(loss_true_folds), np.std(loss_true_folds)),
        "loss_shuf": (np.mean(loss_shuf_folds), np.std(loss_shuf_folds)),
        "r2_true_all": r2_true_folds,
        "r2_shuf_all": r2_shuf_folds,
    }

# ---- 3. Report ----
print()
print("=" * 60)
print("  Isolated 5-Fold CV CEBRA Validation Report (Position)")
print("=" * 60)
for cond_name in ["Tracking", "Playback"]:
    r = all_results[cond_name]
    print(f"\n  --- {cond_name} ---")
    print(f"  True  R2:  {r['r2_true'][0]:.4f} +/- {r['r2_true'][1]:.4f}")
    print(f"  Shuf  R2:  {r['r2_shuf'][0]:.4f} +/- {r['r2_shuf'][1]:.4f}")
    print(f"  True  loss: {r['loss_true'][0]:.4f} +/- {r['loss_true'][1]:.4f}")
    print(f"  Shuf  loss: {r['loss_shuf'][0]:.4f} +/- {r['loss_shuf'][1]:.4f}")
    if r['r2_true'][0] > 0.3 and r['r2_true'][0] > (r['r2_shuf'][0] + 0.2):
        print(f"  => PASSED")
    else:
        print(f"  => FAILED")

# Summary barplot
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
bars = [all_results["Tracking"]["r2_true"][0], all_results["Playback"]["r2_true"][0],
        all_results["Tracking"]["r2_shuf"][0], all_results["Playback"]["r2_shuf"][0]]
errs = [all_results["Tracking"]["r2_true"][1], all_results["Playback"]["r2_true"][1],
        all_results["Tracking"]["r2_shuf"][1], all_results["Playback"]["r2_shuf"][1]]
colors = ['darkred', 'darkblue', 'gray', 'lightgray']
ax.bar(x, bars, yerr=errs, color=colors, capsize=5)
ax.set_xticks(x)
ax.set_xticklabels(['True Tracking', 'True Playback', 'Shuffle Tracking', 'Shuffle Playback'])
ax.set_ylabel("R2 (mean +/- std)")
ax.set_title("SKIEUR: Isolated 5-Fold CV R2 Summary (Position)")
plt.tight_layout(); plt.show()

print()
print(">>> Isolated CV complete.")

In [ ]:
# --- GPU cleanup ---
import gc, torch
del cebra_true, cebra_shuffle
gc.collect(); torch.cuda.empty_cache()
print("GPU memory cleared.")